# XMV-Net — Skeleton

**Owner:** Rafid · **Scope:** the model the project is about (`§3`, `§6`, `§9` of the report)

This notebook is a *runnable skeleton*, not a finished model. Every piece exists and shapes flow
end to end, but three components are deliberately left as **placeholders** — each marked `TODO`.
We replace them one at a time, and the notebook keeps running after every step.

| Component | State | Why |
|---|---|---|
| View splitter | ✅ done | Comes free from `data_loader.VIEW_RULES` |
| Sparse feature gate | ✅ done | 5 lines; the interesting part is the *regularizer*, not the layer |
| Per-view MLP encoder | ✅ done | Standard 2-layer block |
| **Gated attention fusion** | 🔨 placeholder = **mean fusion** | Step 1 to implement |
| **Composite loss** | 🔨 placeholder = **plain BCE** | Step 2 to implement |
| **Training loop** | 🔨 placeholder = **stub** | Step 3 — coordinate with Siam's `train_utils.py` first |

> The mean-fusion placeholder is not throwaway code: it *is* **ablation A1** ("does learned fusion add
> accuracy, or only interpretability?"). Keep it reachable behind a config flag rather than deleting it.

---

## What the data actually looks like

| | |
|---|---|
| Train | 595,000 accounts · 12.68% churn · behaviour Jan–Mar, label = churn in Apr |
| Test | 255,000 accounts · **no label** — all reported metrics come from CV on train |
| Split type | Account-level random (train IDs 1,3,5… / test IDs 2,4,6…), **not** temporal |
| Raw features | 88 numeric + `GENDER`, `REGION` |
| After preprocessing | **127 features across 8 views** (see below) |

**Three data facts that shape the model:**

1. **8 columns are constant zero** in both train and test (`salary_amt_7d/14d`, `salary_amt_ratio_7d/14d`,
   `salary_day_ratio_7d/14d`, `n_salary_7d/14d`). All 8 sit in the *monetary* view. They are dropped —
   a constant column's gate weight is unidentifiable and would inflate the sparsity metrics in `§5`.
2. **44 columns have structural NaNs** (up to 18.9%). They mean *"no transactions in that window"*, not
   *"unknown"* — so they are filled with `0` and paired with a `__isna` indicator assigned to the same view.
3. **The recency cluster is near-definitional.** `rec_ge30 == 1` → **94.2% churn** vs 5.4% otherwise;
   `recency_days` correlates r=0.79 with the label. Every model here will look great for the wrong reason.
   Sameen's **ablation A7** (drop the recency view) is the honest headline — treat pre-A7 numbers as provisional.

**Deviation from the proposal to flag at the next sync:** the proposal specifies **7** views. `GENDER` and
`REGION` belong to none of them, so they form an 8th **demographic** view. Update `§3` of the report to say 8.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import data_loader as dl

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_ROWS = 50_000   # None = full 595k. Keep small while iterating on CPU.

print(f"torch {torch.__version__} | device {DEVICE}")

torch 2.4.0+cpu | device cpu


---
## Step 0 — Data

All CSV handling lives in [`data_loader.py`](data_loader.py) (documented in [`docs/data_loader.md`](docs/data_loader.md)).

The first call runs a two-pass build: pass 1 discovers the schema from train (row count, NaN columns,
constant columns, clip quantiles, skew), pass 2 streams both CSVs into memmapped `.npy` caches.
Roughly 11 s once; every later call is a ~0.02 s memmap open.

In [ ]:
dl.build_cache()

train = dl.load_train(n_rows=SAMPLE_ROWS)
test = dl.load_test()

FEATURE_COLS = train.feature_cols
VIEW_GROUPS = train.view_groups

assert train.feature_cols == test.feature_cols, "FEATURE_COLS drift between train and test"
print(f"train {train.X.shape}  churn={train.y.mean():.4f}   test {test.X.shape}")
print(train.summary().to_string(index=False))

### ⚠️ Read the view sizes before building the model

The views are badly unbalanced: **frequency has 50 features, network has 4.** That matters for two reasons:

- A 50-feature encoder and a 4-feature encoder both produce a 32-d embedding, so attention compares them
  on equal footing even though one saw 12× more input. Expect frequency to dominate attention early.
- The L1 gate penalty `λ_gate · Σ‖m⁽ᵛ⁾‖₁` sums over features, so it penalises the frequency view ~12× harder
  than network for the same *average* gate value. **Consider normalising the gate penalty by view width**
  (`mean` instead of `sum`) — otherwise A5's "cost of interpretability" is really just "cost of being wide".

Options to raise at the sync: merge `network` into `frequency` (7 views), or keep 8 and report the imbalance honestly.

In [ ]:
# Sanity: views are disjoint and cover every feature exactly once.
_all = np.concatenate(list(VIEW_GROUPS.values()))
assert len(_all) == len(np.unique(_all)) == len(FEATURE_COLS)

# The leak you will be asked about in the viva.
i = FEATURE_COLS.index("rec_ge30")
flag = np.asarray(train.X[:, i]) > 0
print(f"rec_ge30=1 -> churn {train.y[flag].mean():.3f}  (n={flag.sum():,})")
print(f"rec_ge30=0 -> churn {train.y[~flag].mean():.3f}  (n={(~flag).sum():,})")

---
## Step 1 — Folds and standardisation

**Placeholder folds.** Sameen owns the frozen 5-fold split; it does not exist yet. `make_folds` loads the
frozen file when you pass `frozen_path`, and otherwise generates a seeded `StratifiedKFold` with a loud warning.

> 🔁 **Single swap point.** When the frozen file lands, set `FROZEN_FOLDS` below and change nothing else.
> Numbers produced before that point are **not** comparable to Siam's or Tausif's.

**Standardisation is fitted per fold, on training rows only.** Fitting on all rows leaks validation
distribution into training — mild, but free to avoid, and this project is partly *about* leakage discipline.

In [ ]:
FROZEN_FOLDS = None   # <-- swap point: set to Sameen's frozen fold .npy path

folds = dl.make_folds(train.y, n_splits=5, seed=SEED, frozen_path=FROZEN_FOLDS)

FOLD = 0
tr_idx = np.where(folds != FOLD)[0]
va_idx = np.where(folds == FOLD)[0]

mean, std = dl.fit_standardizer(train.X, tr_idx)
Xtr = dl.apply_standardizer(np.asarray(train.X[tr_idx]), mean, std)
Xva = dl.apply_standardizer(np.asarray(train.X[va_idx]), mean, std)
ytr, yva = train.y[tr_idx].astype(np.float32), train.y[va_idx].astype(np.float32)

print(f"fold {FOLD}: train {Xtr.shape} churn={ytr.mean():.4f} | val {Xva.shape} churn={yva.mean():.4f}")

---
## Step 2 — Architecture

```
        x  (B, 127)
         │
         ├── split by view ──►  x⁽ᵛ⁾  for v = 1..8
         │                       │
         │                  ┌────┴─────┐
         │                  │ gate m⁽ᵛ⁾│   m⁽ᵛ⁾ = σ(w⁽ᵛ⁾ / τ)      ← feature-level explanation
         │                  └────┬─────┘
         │                       │  x⁽ᵛ⁾ ⊙ m⁽ᵛ⁾
         │                  ┌────┴─────┐
         │                  │ encoder  │   Linear→BN→GELU→Drop ×2  (·→64→32)
         │                  └────┬─────┘
         │                       │  h⁽ᵛ⁾ (B, 32)
         │              ┌────────┼────────┐
         │              │                 │
         │        ┌─────┴─────┐    ┌──────┴──────┐
         │        │ aux head  │    │  attention  │   aᵥ ∝ exp(wᵀ(tanh(Vh⁽ᵛ⁾) ⊙ σ(Uh⁽ᵛ⁾)))
         │        └─────┬─────┘    └──────┬──────┘   ← view-level explanation
         │              │                 │  z = Σᵥ aᵥ h⁽ᵛ⁾
         │       aux logits (8)     ┌─────┴─────┐
         │                          │   head    │
         │                          └─────┬─────┘
         │                             logit
```

### Two design points worth understanding now

**Why `tanh ⊙ σ` gating instead of plain dot-product attention?** With only 8 items, plain
`softmax(wᵀh)` saturates — one view takes ~all the mass within a few epochs and never recovers. The Ilse
*et al.* gated form gives the scoring function a multiplicative bypass (`σ(Uh)`), which keeps gradients
alive for low-scoring views.

**Why the auxiliary heads are not optional.** Gradient reaching encoder *v* through the fused path is
scaled by `aᵥ`. A view that starts with low attention gets a small gradient → learns slowly → stays
uninformative → keeps low attention. That is a self-reinforcing collapse. The aux heads give every encoder
an **unconditional** gradient path, independent of `aᵥ`. Ablation **A4** (`λ_aux = 0`) is the row that
demonstrates this — expect it to look visibly worse, and that is the point.

In [ ]:
class FeatureGate(nn.Module):
    """Learned soft feature mask m = sigmoid(w / tau), applied elementwise."""

    def __init__(self, n_features: int, tau: float = 1.0, init: float = 1.0):
        super().__init__()
        self.w = nn.Parameter(torch.full((n_features,), init))
        self.tau = tau

    def mask(self) -> torch.Tensor:
        return torch.sigmoid(self.w / self.tau)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.mask()


class ViewEncoder(nn.Module):
    """Gate -> 2-layer MLP -> embedding of size `out_dim`."""

    def __init__(self, in_dim: int, hidden: int = 64, out_dim: int = 32,
                 dropout: float = 0.1, tau: float = 1.0):
        super().__init__()
        self.gate = FeatureGate(in_dim, tau=tau)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim), nn.BatchNorm1d(out_dim), nn.GELU(), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.gate(x))

### 🔨 TODO #1 — Gated attention fusion

Currently a **uniform-mean placeholder**, so the rest of the notebook runs. Replace it with:

$$e_v = \mathbf{w}^\top\big(\tanh(V h^{(v)}) \odot \sigma(U h^{(v)})\big), \qquad a_v = \mathrm{softmax}_v(e_v / T)$$

where $V, U \in \mathbb{R}^{A \times d}$ and $\mathbf{w} \in \mathbb{R}^{A}$ ($A$ = attention dim, try 32).
$T$ is the annealing temperature: start at 1.0, decay toward ~0.5 so attention sharpens late in training
rather than collapsing early.

Input `H` is `(B, n_views, d)`; return `(z, a)` with `z` of shape `(B, d)` and `a` of shape `(B, n_views)`
summing to 1 along dim 1.

In [ ]:
class GatedAttention(nn.Module):
    """PLACEHOLDER: uniform mean fusion. This is also ablation A1."""

    def __init__(self, dim: int, attn_dim: int = 32):
        super().__init__()
        self.dim = dim
        self.attn_dim = attn_dim
        # TODO #1: self.V = nn.Linear(dim, attn_dim, bias=False)
        #          self.U = nn.Linear(dim, attn_dim, bias=False)
        #          self.w = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, H: torch.Tensor, temperature: float = 1.0):
        B, V, _ = H.shape
        # TODO #1: e = self.w(torch.tanh(self.V(H)) * torch.sigmoid(self.U(H))).squeeze(-1)
        #          a = torch.softmax(e / temperature, dim=1)
        a = H.new_full((B, V), 1.0 / V)
        z = (a.unsqueeze(-1) * H).sum(dim=1)
        return z, a

In [ ]:
class XMVNet(nn.Module):
    """Multi-view encoder -> gated attention fusion -> prediction head, plus per-view aux heads."""

    def __init__(self, view_groups: dict[str, np.ndarray], hidden: int = 64, emb_dim: int = 32,
                 attn_dim: int = 32, dropout: float = 0.1, tau: float = 1.0, use_attention: bool = True):
        super().__init__()
        self.view_names = list(view_groups.keys())
        self.use_attention = use_attention

        for name, idx in view_groups.items():
            self.register_buffer(f"idx_{name}", torch.as_tensor(np.asarray(idx), dtype=torch.long))

        self.encoders = nn.ModuleDict({
            name: ViewEncoder(len(idx), hidden, emb_dim, dropout, tau)
            for name, idx in view_groups.items()
        })
        self.aux_heads = nn.ModuleDict({name: nn.Linear(emb_dim, 1) for name in view_groups})
        self.fusion = GatedAttention(emb_dim, attn_dim)
        self.head = nn.Sequential(nn.Linear(emb_dim, 32), nn.GELU(), nn.Linear(32, 1))

    def forward(self, x: torch.Tensor, temperature: float = 1.0) -> dict:
        embeddings, aux_logits = [], []
        for name in self.view_names:
            h = self.encoders[name](x[:, getattr(self, f"idx_{name}")])
            embeddings.append(h)
            aux_logits.append(self.aux_heads[name](h).squeeze(-1))

        H = torch.stack(embeddings, dim=1)
        z, a = self.fusion(H, temperature)

        return {
            "logit": self.head(z).squeeze(-1),
            "attn": a,
            "aux_logits": torch.stack(aux_logits, dim=1),
            "gates": {n: self.encoders[n].gate.mask() for n in self.view_names},
        }

In [ ]:
model = XMVNet(VIEW_GROUPS).to(DEVICE)

xb = torch.from_numpy(Xtr[:8]).to(DEVICE)
out = model(xb)

print(f"params: {sum(p.numel() for p in model.parameters()):,}")
print(f"logit      {tuple(out['logit'].shape)}")
print(f"attn       {tuple(out['attn'].shape)}  rowsum={out['attn'].sum(1)[0].item():.4f}")
print(f"aux_logits {tuple(out['aux_logits'].shape)}")
for n, g in out["gates"].items():
    print(f"  gate[{n:<12}] dim={tuple(g.shape)} mean={g.mean().item():.3f}")

---
## Step 3 — Loss

### 🔨 TODO #2 — the composite objective

$$\mathcal{L} = \underbrace{\mathcal{L}_{BCE}(\text{fused})}_{\text{primary}} + \lambda_{aux}\sum_v \mathcal{L}_{BCE}(\text{head}_v) + \lambda_{ent} H(a) + \lambda_{gate}\sum_v \lVert m^{(v)} \rVert_1$$

Currently only the first term is implemented. Add the rest one at a time and watch the diagnostics move.

| Term | Suggested λ | What it buys | Failure mode if wrong |
|---|---|---|---|
| primary BCE | — | calibrated probabilities (needed for `§8` ECE/Brier) | — |
| aux BCE | 0.3 | unconditional gradient to every encoder | λ too high → fused head under-trains |
| attention entropy | small, **annealed 0 → λ** | decisive view attributions | applied from epoch 0 → premature collapse |
| gate L1 | tune | within-view sparsity | too low → gates saturate at 1 (no explanation) |

**Two sign conventions that are easy to get backwards:**
- $H(a) = -\sum_v a_v \log a_v$ is **maximal** for uniform attention. You want attention *sharp*, so
  you **add** $+\lambda_{ent} H(a)$ to minimise entropy. Adding $-\lambda_{ent}H(a)$ pushes toward uniform — the opposite.
- The gate L1 uses `mask()` (post-sigmoid, in [0,1]), **not** the raw logit `w`. Penalising `w` drives it to 0,
  which is `sigmoid(0) = 0.5` — a half-open gate, not a closed one.

In [ ]:
def xmv_loss(out: dict, y: torch.Tensor, lam_aux: float = 0.3, lam_ent: float = 0.0,
             lam_gate: float = 0.0, gate_reduction: str = "mean") -> tuple[torch.Tensor, dict]:
    """PLACEHOLDER: primary BCE only. Returns (loss, per-term dict for logging)."""
    bce = F.binary_cross_entropy_with_logits(out["logit"], y)

    # TODO #2a: aux = F.binary_cross_entropy_with_logits(
    #               out["aux_logits"], y.unsqueeze(1).expand_as(out["aux_logits"]))
    # TODO #2b: a = out["attn"].clamp_min(1e-8); ent = -(a * a.log()).sum(1).mean()
    # TODO #2c: gate = sum of m.sum() or m.mean() per view  <- see the view-width note above
    aux = torch.zeros((), device=bce.device)
    ent = torch.zeros((), device=bce.device)
    gate = torch.zeros((), device=bce.device)

    total = bce + lam_aux * aux + lam_ent * ent + lam_gate * gate
    return total, {"bce": bce.item(), "aux": aux.item(), "ent": ent.item(), "gate": gate.item()}


yb = torch.from_numpy(ytr[:8]).to(DEVICE)
loss, parts = xmv_loss(out, yb)
loss.backward()
print(f"loss {loss.item():.4f}  {parts}")
print("grad reaches every encoder:",
      all(model.encoders[n].gate.w.grad is not None for n in model.view_names))

---
## Step 4 — Diagnostics (wire these before the first real training run)

Both documented failure modes are silent — training loss looks fine while the *explanation*, which is the
entire point of the project, quietly dies. Log these **every epoch** from run #1, not after something breaks.

| Diagnostic | Healthy | Alarm | Fix |
|---|---|---|---|
| mean attention entropy `H(a)` | rises toward `ln(8) ≈ 2.08` early, falls slowly | **< 0.3 nats before epoch 5** | λ_ent too high, or aux heads mis-wired |
| per-view mean gate `m⁽ᵛ⁾` | spread across (0,1) | **all ≈ 1.0** (saturated) | raise λ_gate an order of magnitude |
| per-view attention **variance** across customers | clearly > 0 | ≈ 0 (e.g. `0.34 ± 0.01`) | model has collapsed to fixed global weights — the "per-customer attention" claim would be false |

That third row is Tausif's sanity check from their brief, but it is much cheaper to catch here than in week 3.

In [ ]:
@torch.no_grad()
def diagnostics(model: XMVNet, X: np.ndarray, n: int = 4096, batch: int = 1024) -> dict:
    model.eval()
    attn = []
    for s in range(0, min(n, len(X)), batch):
        xb = torch.from_numpy(X[s:s + batch]).to(DEVICE)
        attn.append(model(xb)["attn"].cpu())
    A = torch.cat(attn)

    ent = -(A.clamp_min(1e-8) * A.clamp_min(1e-8).log()).sum(1).mean().item()
    gates = {n_: g.mean().item() for n_, g in model(torch.from_numpy(X[:2]).to(DEVICE))["gates"].items()}
    model.train()
    return {
        "attn_entropy": ent,
        "attn_entropy_max": float(np.log(A.shape[1])),
        "attn_mean": dict(zip(model.view_names, A.mean(0).tolist())),
        "attn_std": dict(zip(model.view_names, A.std(0).tolist())),
        "gate_mean": gates,
    }


d = diagnostics(model, Xtr)
print(f"H(a) = {d['attn_entropy']:.3f} / {d['attn_entropy_max']:.3f} max")
print(pd.DataFrame({"attn_mean": d["attn_mean"], "attn_std": d["attn_std"],
                    "gate_mean": d["gate_mean"]}).round(4).to_string())
print("\n(attn_std is all-zero right now — that is the mean-fusion placeholder, as expected.)")

---
## Step 5 — Training

### 🔨 TODO #3 — do **not** write this loop yet

Per your brief and Siam's: you both need PyTorch + AMP + early-stopping-on-fold-val-AUC + config logging.
Siam owns `train_utils.py` and is meant to deliver it day 2–3. **Talk to them before filling this in** —
two training loops for the same underlying loop is wasted effort, and it makes your numbers and theirs
diverge for reasons nobody can later reconstruct.

What XMV-Net needs on top of a generic harness — flag these to Siam so the interface accommodates them:

1. The loss takes the **whole output dict**, not just logits (aux heads, attention, gates).
2. A **per-epoch scalar schedule** hook — temperature `T` and `λ_ent` both anneal.
3. A **per-epoch diagnostics hook** — the table above must be logged alongside loss/AUC.

Below is a minimal in-memory tensor loader so you can smoke-test forward/backward today.

**Before the first full run: overfit a 50K sample.** If XMV-Net cannot drive training loss to ~0 on 50K rows
with regularisation off, something is structurally wrong — and you want to find that on day 3, not day 10.

**Keep A1/A4/A5/A6 as config flags on this one class.** Forked copies of the model file drift apart within days.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int = 1024, shuffle: bool = True) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(np.ascontiguousarray(X)), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)


def train_one_fold(model, Xtr, ytr, Xva, yva, epochs=10, lr=1e-3, **loss_kwargs):
    """STUB. Fill in once Siam's train_utils.py exists -- see TODO #3."""
    raise NotImplementedError("TODO #3: build on train_utils.py, do not write a second training loop")


# Smoke test: 20 steps of real optimisation. Loss should fall.
model = XMVNet(VIEW_GROUPS).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loader = make_loader(Xtr, ytr)

model.train()
for step, (xb, yb) in enumerate(loader):
    if step >= 20:
        break
    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    loss, parts = xmv_loss(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 5 == 0:
        print(f"step {step:>3}  loss {loss.item():.4f}")

---
## Where we are, and what is next

**Working now:** loader + cache · view splitter · feature gates · per-view encoders · aux heads ·
forward pass · diagnostics · smoke-tested backward pass.

**Next, in order:**

1. **TODO #1** — gated attention (`tanh ⊙ σ`). Verify `attn_std > 0` afterwards; if it is still ~0, attention
   is not per-customer and the headline figure would be a lie.
2. **TODO #2** — composite loss, one term at a time, watching the diagnostics after each.
3. **Overfit 50K** with all λ = 0. Must reach ~0 training loss.
4. **TODO #3** — training loop on Siam's `train_utils.py`.
5. Full 5-fold on the **frozen** folds, then light Optuna (≤25 trials, 1 fold).
6. Ablations A1/A4/A5/A6 as config flags.
7. Ship checkpoints + attention/gate outputs to Tausif — **target end of week 2, this is the tightest
   dependency in the project.**

**Raise at the next sync:**
- 8 views, not 7 (`GENDER`/`REGION` → demographic view). `§3` of the report needs updating.
- View width imbalance 50 vs 4 — merge `network` into `frequency`, or normalise the gate penalty by width?
- 8 constant-zero columns dropped; **all 8 were monetary**, leaving that view with 7 real features.
- Need Sameen's frozen folds. Everything above is on placeholder folds and is **not** yet comparable.
- Test set has no labels → no test-set metric is possible; all `§8` numbers come from CV on train.
  Confirm whether a `submission.csv` of test predictions is a deliverable at all.

**Environment note:** local torch is CPU-only, and `captum` / `shap` / `pyarrow` are not installed. Fine for
this skeleton; install on Kaggle before week 3's explainability work.